# 02 — Feature engineering : Superstore Sales

Ce notebook prépare les lignes de vente pour une première modélisation. Il transforme les dates, crée des variables calendaires, encode les catégories et met à l'échelle les variables numériques. Les transformations sont ajustées uniquement sur le jeu d'entraînement pour éviter toute fuite d'information vers le jeu de test.

**Cible choisie : `Sales`.** Elle est conservée à part des variables explicatives et ajoutée aux fichiers de sortie comme valeur à prédire. Le fichier source `train.csv` n'est jamais modifié.

Le CSV Superstore disponible ne contient pas `Profit`, `Quantity` ou `Discount` : aucune variable ne sera inventée à partir de ces champs.

## 1. Importer les outils et charger les données

Les dates du fichier sont au format jour/mois/année. `dayfirst=True` indique explicitement cette convention lors de leur conversion. Les chemins sont relatifs au dossier `notebooks/`.

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Le notebook se trouve dans notebooks/ ; les données et sorties sont dans data/.
DATA_DIR = Path("../data")
INPUT_FILE = DATA_DIR / "train.csv"

# Le fichier ne sera pas écrasé : les sorties portent de nouveaux noms.
sales = pd.read_csv(INPUT_FILE)

print(f"Dimensions du fichier : {sales.shape[0]} lignes et {sales.shape[1]} colonnes")
display(sales.head())

Dimensions du fichier : 9800 lignes et 18 colonnes


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


## 2. Vérifier les colonnes et préparer les valeurs manquantes

On contrôle d'abord que les champs nécessaires sont présents. Les lignes sans cible `Sales` ou sans date de commande/expédition exploitable ne peuvent pas servir à cette préparation : elles sont retirées et leur nombre est affiché. Pour les catégories, une valeur manquante est remplacée par `Unknown` afin de ne pas supprimer inutilement des ventes.

In [2]:
required_columns = {
    "Order Date", "Ship Date", "Sales", "Ship Mode", "Segment",
    "Country", "City", "State", "Postal Code", "Region",
    "Category", "Sub-Category",
}
missing_columns = required_columns.difference(sales.columns)
if missing_columns:
    raise ValueError(f"Colonnes nécessaires absentes du CSV : {sorted(missing_columns)}")

# Garder une copie de travail pour préserver le tableau chargé initialement.
sales_fe = sales.copy()

# Convertir les dates en tenant compte du format jour/mois/année du fichier.
# errors="coerce" transforme les dates invalides en NaT, qui seront comptées ci-dessous.
sales_fe["Order Date"] = pd.to_datetime(
    sales_fe["Order Date"], dayfirst=True, errors="coerce"
)
sales_fe["Ship Date"] = pd.to_datetime(
    sales_fe["Ship Date"], dayfirst=True, errors="coerce"
)
sales_fe["Sales"] = pd.to_numeric(sales_fe["Sales"], errors="coerce")

invalid_rows = sales_fe[["Order Date", "Ship Date", "Sales"]].isna().any(axis=1)
print(f"Lignes retirées (date ou Sales manquante/invalide) : {invalid_rows.sum()}")
sales_fe = sales_fe.loc[~invalid_rows].copy()

print("Valeurs manquantes dans les champs catégoriels avant remplacement :")
categorical_columns = [
    "Ship Mode", "Segment", "Country", "City", "State", "Postal Code",
    "Region", "Category", "Sub-Category",
]
display(sales_fe[categorical_columns].isna().sum())

# Postal Code est une catégorie géographique, pas une mesure numérique.
# Le convertir en texte évite que le modèle interprète, par exemple, 90036
# comme une quantité deux fois plus grande que 45018.
sales_fe["Postal Code"] = sales_fe["Postal Code"].astype("string")
sales_fe[categorical_columns] = sales_fe[categorical_columns].fillna("Unknown")

Lignes retirées (date ou Sales manquante/invalide) : 0
Valeurs manquantes dans les champs catégoriels avant remplacement :


Ship Mode        0
Segment          0
Country          0
City             0
State            0
Postal Code     11
Region           0
Category         0
Sub-Category     0
dtype: int64

## 3. Créer des variables dérivées des dates

`Order Year`, `Order Month` et `Order Weekday` décrivent le moment de la commande et peuvent aider à repérer des tendances temporelles. `Shipping Days` mesure le nombre de jours entre commande et expédition : c'est une nouvelle variable métier, utile pour analyser la logistique.

**Attention à la disponibilité de l'information :** `Shipping Days` et `Ship Date` ne sont connus qu'après l'expédition. Ils ne sont donc pas utilisés comme variables explicatives dans le modèle ci-dessous, qui est conçu comme une prédiction au moment de la commande. Utiliser le délai pour ce scénario créerait une fuite d'information. Le délai reste présent dans le tableau enrichi pour l'analyse.

In [3]:
# Délai logistique en jours ; il est conservé pour l'analyse, pas pour prédire Sales.
sales_fe["Shipping Days"] = (
    sales_fe["Ship Date"] - sales_fe["Order Date"]
).dt.days

# Variables calendaires connues à la date de la commande.
sales_fe["Order Year"] = sales_fe["Order Date"].dt.year
sales_fe["Order Month"] = sales_fe["Order Date"].dt.month
# Convention pandas : lundi = 0 et dimanche = 6.
sales_fe["Order Weekday"] = sales_fe["Order Date"].dt.dayofweek

display(
    sales_fe[["Order Date", "Ship Date", "Shipping Days", "Order Year", "Order Month", "Order Weekday"]].head()
)
print("Délais d'expédition négatifs :", (sales_fe["Shipping Days"] < 0).sum())

,Order Date,Ship Date,Shipping Days,Order Year,Order Month,Order Weekday
0,2017-11-08,2017-11-11,3,2017,11,2
1,2017-11-08,2017-11-11,3,2017,11,2
2,2017-06-12,2017-06-16,4,2017,6,0
3,2016-10-11,2016-10-18,7,2016,10,1
4,2016-10-11,2016-10-18,7,2016,10,1


Délais d'expédition négatifs : 0


## 4. Définir la cible et les variables du modèle

`Sales` est la valeur à prédire. Les identifiants (`Row ID`, `Order ID`, `Customer ID`, `Product ID`) et les noms (`Customer Name`, `Product Name`) sont écartés : ils identifient une ligne, un client ou un produit, mais ne constituent pas de mesures continues. Les dates brutes sont remplacées par leurs variables calendaires. Les champs géographiques et commerciaux sont encodés comme catégories.

Les champs `City` et `Postal Code` peuvent créer beaucoup de colonnes après encodage. Ils sont gardés ici pour une première version complète ; si le modèle généralise mal, teste une version sans eux.

In [4]:
target_column = "Sales"

# Les variables catégorielles sans ordre naturel seront encodées en one-hot.
categorical_features = [
    "Ship Mode", "Segment", "Country", "City", "State", "Postal Code",
    "Region", "Category", "Sub-Category",
]

# Ces variables numériques sont connues au moment de la commande.
# Shipping Days est volontairement exclu car il est connu après la vente.
numeric_features = ["Order Year", "Order Month", "Order Weekday"]
feature_columns = categorical_features + numeric_features

X = sales_fe[feature_columns].copy()
y = sales_fe[target_column].copy()

print("Variables explicatives :", len(feature_columns))
print("Lignes prêtes à être séparées :", len(X))
print("Cible conservée à part :", target_column)

Variables explicatives : 12
Lignes prêtes à être séparées : 9800
Cible conservée à part : Sales


## 5. Séparer entraînement et test, puis ajuster les transformations

On sépare les lignes avant d'apprendre les moyennes/écarts-types du scaler et la liste des catégories du one-hot encoder. Ainsi, le jeu de test ne participe pas à l'ajustement des transformations. `handle_unknown="ignore"` permet aussi de traiter une catégorie présente dans le test mais absente de l'entraînement.

Les colonnes numériques sont standardisées ; les catégories sont transformées en indicateurs 0/1. Les identifiants et les colonnes dates brutes ne passent pas dans le modèle.

In [5]:
# Un découpage aléatoire reproductible : 80 % pour l'entraînement, 20 % pour le test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Chaque type de colonne reçoit une transformation adaptée.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ],
    remainder="drop",
)

# Important : fit_transform sur l'entraînement seulement ; test utilise transform.
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# OneHotEncoder renvoie souvent une matrice creuse. Pour écrire un CSV lisible,
# on la convertit en tableau dense à ce stade (le fichier Superstore reste modeste).
if hasattr(X_train_encoded, "toarray"):
    X_train_encoded = X_train_encoded.toarray()
if hasattr(X_test_encoded, "toarray"):
    X_test_encoded = X_test_encoded.toarray()

encoded_feature_names = preprocessor.get_feature_names_out()
X_train_ready = pd.DataFrame(
    X_train_encoded, columns=encoded_feature_names, index=X_train.index
)
X_test_ready = pd.DataFrame(
    X_test_encoded, columns=encoded_feature_names, index=X_test.index
)

# Ajouter la cible séparément : les features ne contiennent pas Sales.
train_ready = X_train_ready.copy()
train_ready[target_column] = y_train
test_ready = X_test_ready.copy()
test_ready[target_column] = y_test

print("Dimensions train transformé :", train_ready.shape)
print("Dimensions test transformé :", test_ready.shape)
print("Colonnes contenant du texte dans train :", train_ready.select_dtypes(include="object").columns.tolist())
display(train_ready.head())

Dimensions train transformé : (7840, 1198)
Dimensions test transformé : (1960, 1198)
Colonnes contenant du texte dans train : []


,numeric__Order Year,numeric__Order Month,numeric__Order Weekday,categorical__Ship Mode_First Class,categorical__Ship Mode_Same Day,categorical__Ship Mode_Second Class,categorical__Ship Mode_Standard Class,categorical__Segment_Consumer,categorical__Segment_Corporate,categorical__Segment_Home Office,...,categorical__Sub-Category_Fasteners,categorical__Sub-Category_Furnishings,categorical__Sub-Category_Labels,categorical__Sub-Category_Machines,categorical__Sub-Category_Paper,categorical__Sub-Category_Phones,categorical__Sub-Category_Storage,categorical__Sub-Category_Supplies,categorical__Sub-Category_Tables,Sales
3984,0.243356,-0.250118,0.463114,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,48.940
4824,0.243356,0.969173,-0.910287,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,127.554
1414,1.135739,-2.079054,1.378715,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,239.970
540,-1.541409,-1.774231,1.378715,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,468.900
6522,1.135739,-2.079054,-0.910287,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,56.070


## 6. Enregistrer les résultats

Deux fichiers distincts permettent d'entraîner et d'évaluer un modèle sans mélanger les partitions. `Sales` reste la cible dans les deux fichiers ; toutes les autres colonnes sont numériques. Le préprocesseur est aussi sauvegardé pour réutiliser exactement les mêmes catégories et paramètres de mise à l'échelle sur de nouvelles données.

In [6]:
train_output = DATA_DIR / "engineered_train.csv"
test_output = DATA_DIR / "engineered_test.csv"
preprocessor_output = DATA_DIR / "feature_preprocessor.joblib"

# index=False évite d'ajouter l'index pandas comme une colonne supplémentaire.
train_ready.to_csv(train_output, index=False)
test_ready.to_csv(test_output, index=False)
joblib.dump(preprocessor, preprocessor_output)

print("Fichiers créés :")
print(train_output)
print(test_output)
print(preprocessor_output)

Fichiers créés :
../data/engineered_train.csv
../data/engineered_test.csv
../data/feature_preprocessor.joblib


## Résumé et précautions

- `Sales` est la cible ; ne la passe pas au modèle comme variable explicative.
- Les variables catégorielles ont été encodées en nombres et les variables calendaires mises à l'échelle.
- `Shipping Days` a été créé et documenté, mais exclu du modèle car il n'est pas disponible au moment de la commande.
- `engineered_train.csv` et `engineered_test.csv` sont prêts pour l'étape de modélisation ; `feature_preprocessor.joblib` conserve les transformations ajustées sur l'entraînement.
- Si la question métier est une prévision dans le futur plutôt qu'une prédiction de chaque ligne de vente, il faudra aussi faire un découpage chronologique (et non aléatoire) pour respecter l'ordre des dates.

Les champs `City` et `Postal Code` donnent de nombreuses colonnes encodées. Vérifie leur utilité lors de la validation du modèle ; ils peuvent être retirés de `categorical_features` si nécessaire.